# 軟體設計與開發 期中考 B 卷：校園咖啡廳預約取餐系統

考試時間 3 小時。請依序完成：規格書 → 產生程式 → 測試 → 交件。

## 考試規則

1. 個人獨立完成；可使用任何 AI 工具；**禁止與其他同學討論或交換檔案**。
2. 第 1 格由老師提供，**不可修改**。唯一例外：若 `openai/gpt-oss-20b` 額度用完，可將 `MODEL` 改為 `"openai/gpt-oss-120b"`，並在規格書最後註明。
3. Colab 密鑰名稱一律為 `GROQ_API_KEY`，名稱錯誤扣分。程式或輸出中出現金鑰，「交件」一項以 0 分計。
4. 測試數量：AI 解讀測試 8～10 句、流程測試 8～10 步，**不要超過上限**。
5. **為避免額度用完**，除錯時請請先"一格一格"除錯；等確認所有code cell都沒問題，在執行「重新啟動並執行全部」→上傳交卷。

## 情境

系上咖啡廳想讓同學用一句話**預約飲品、取消、查詢剩餘庫存**，並讓店長看到今天的**銷售統計**。

**初始資料：**

| 品項 | 單價（元） | 今日庫存（杯） |
|---|---|---|
| 美式咖啡 | 50 | 5 |
| 拿鐵 | 65 | 4 |
| 抹茶歐蕾 | 70 | 3 |

## 基本需求（90 分）

**B1 規格書（20 分）：** 完成筆記本中的規格書 7 節。

**B2 系統功能（40 分）：**
- 兩張資料表：菜單、訂單；每次執行都重建並放入初始資料。
- AI 從一句話中抽出「動作」與「品項」。動作有：訂購、取消、查詢。無法判斷時，回覆目前有哪些品項。
- 訂購：庫存不足時拒絕；成功時回覆金額。
- 取消：把該學生該品項最新一筆訂單的狀態改為「已取消」並釋出庫存；查不到訂單時告知使用者。
- 查詢：回覆該品項的剩餘杯數。
- 訂購與取消需要姓名，查詢不需要。
- Gradio 網頁（輸入需求與姓名，輸出系統回應）。

**B3 測試（20 分）：** AI 解讀測試 8～10 句（正常、錯誤、邊界都要有），並印出通過幾句；流程測試 8～10 步，最後顯示資料表。

**B4 交件（10 分）：** 依「交件方式」完成。

## 進階需求（30 分）

**A1 數量欄位（10 分）：** AI 再抽出第 3 個欄位「數量」（例如「兩杯拿鐵」＝2），沒說時視為 1 杯。數量不在 1～3 之間時由程式拒絕。庫存與金額改以**數量**計算。

**A2 每人上限（10 分）：** 每位學生今天最多訂 **3 杯**（所有品項加總，已取消的不計）。

**A3 統計查詢（10 分）：** 使用者說「今天各品項賣得怎樣」之類的話時，用 **JOIN 與 GROUP BY** 列出每個品項的單價、庫存、已售杯數、剩餘杯數、營收。**沒有人訂的品項也必須列出。**

## 交件方式

1. 選單「執行階段 → 重新啟動並執行全部」，確認沒有紅色錯誤，且每格執行編號連續遞增。
2. 選單「檔案 → 下載 → 下載 .ipynb」，檔名改為 `學號_期中考B.ipynb`。
3. 在電腦上建立資料夾，放入該檔案，在 GitHub 建立**公開且空的** repo（不勾選 README），然後：

```bash
git config --global user.name "你的名字"
git config --global user.email "你的 GitHub 信箱"
git init
git add 學號_期中考B.ipynb
git commit -m "期中考 B 卷"
git branch -M main
git remote add origin https://github.com/你的帳號/你的repo.git
git push -u origin main
```

4. 到作業區**同時上傳**：repo 連結、`.ipynb` 檔案本身。評分以作業區的檔案為準。

## 配分

| 層級 | 代號 | 項目 | 分數 |
|---|---|---|---|
| 基本 | B1 | 規格書 7 節完整 | 20 |
| 基本 | B2 | 系統功能 | 40 |
| 基本 | B3 | 測試與執行結果 | 20 |
| 基本 | B4 | 交件（git push＋作業區） | 10 |
| 進階 | A1 | 第 3 個欄位（數量） | 10 |
| 進階 | A2 | 每人上限規則 | 10 |
| 進階 | A3 | 統計查詢 | 10 |

# 📋 規格書（請直接在這一格修改）

學號：411312676　姓名：宋媫　卷別：B

## 1. 系統目的（一句話）
透過 AI 自然語言理解與 SQLite 資料庫，提供校園咖啡廳線上預約、取消、庫存查詢、個人限額檢查及銷售統計功能。

## 2. 使用者會怎麼說（至少 5 句）
1.我要訂兩杯拿鐵，我是王小明
2.查詢美式咖啡剩餘庫存
3.取消剛剛的美式咖啡，我是王小明
4.幫我訂一杯抹茶歐蕾，我是林小美
5.今天各品項賣得怎樣？

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item_name | TEXT (Primary Key) | 品項名稱 |
| menu | price | INTEGER | 單價（元） |
| menu | stock | INTEGER | 今日庫存（杯） |
| orders | id | INTEGER (Primary Key AUTOINCREMENT) | 訂單編號 |
| orders | student_name | TEXT | 學生姓名 |
| orders | item_name | TEXT | 品項名稱 |
| orders | quantity | INTEGER | 數量 |
| orders | status | TEXT | 狀態（正常 / 已取消） |
| orders | created_at | TIMESTAMP | 訂購時間 |

初始資料：
美式咖啡｜單價：50 元｜今日庫存：5 杯
拿鐵｜單價：65 元｜今日庫存：4 杯
抹茶歐蕾｜單價：70 元｜今日庫存：3 杯

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,品項,數量`
- 動作只能是：訂購、取消、查詢、統計
- 品項只能是：美式咖啡、拿鐵、抹茶歐蕾、未知
- 數量的規則：由正規表示式萃取數字，若未提及則預設為 1
- 無法判斷時輸出：未知,未知,1
- AI 不可以做的事：直接修改資料庫、忽略安全性檢查、回傳非指定格式

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購 | 品項存在且庫存充足、個人今日總杯數加新訂單小於等於 3、數量 1~3 | 預約成功，總金額 $X 元 | orders 新增一筆，menu 庫存扣除 |
| 訂購失敗 | 庫存不足 或 超過個人限額 (3杯) 或 數量不符 | 拒絕原因（庫存不足/超過個人每日3杯上限/數量錯誤） | 資料庫不變 |
| 取消 | 該學生該品項存在最新「正常」訂單 | 取消成功，已釋放庫存 | orders 狀態改為「已取消」，menu 庫存加回 |
| 取消失敗 | 找不到對應訂單 | 查無您的有效訂單 | 資料庫不變 |
| 查詢 | 品項存在 | 顯示該品項剩餘杯數 | 資料庫不變 |
| 統計 | 詢問銷售狀況 | 列出各品項單價、庫存、已售杯數、剩餘杯數、營收 | 資料庫不變 |

## 6. 例外情況（至少 3 個）
1.使用者輸入未販售的品項或語意不明，系統提示可用品項。
2.訂購數量超過 3 杯或小於 1 杯，系統拒絕並提示數量限制 (1~3)。
3.學生嘗試取消沒有訂過的品項，系統回報查無訂單。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**

| # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
|---|---|---|---|---|---|
| 1 | 正常 | 我要訂兩杯拿鐵，我是陳同學 | 訂購 | 拿鐵 | 2 |
| 2 | 正常 | 查詢抹茶歐蕾剩餘庫存 | 查詢 | 抹茶歐蕾 | 1 |
| 3 | 正常 | 取消美式咖啡，我是王小明 | 取消 | 美式咖啡 | 1 |
| 4 | 正常 | 今天各品項賣得怎樣？ | 統計 | 未知 | 1 |
| 5 | 邊界 | 我要訂3杯美式咖啡，我是張三 | 訂購 | 美式咖啡 | 3 |
| 6 | 錯誤 | 我想要一杯珍奶，我是李四 | 訂購 | 未知 | 1 |
| 7 | 邊界 | 訂四杯拿鐵，我是王五 | 訂購 | 拿鐵 | 4 |
| 8 | 錯誤 | 隨便亂講話測試 | 未知 | 未知 | 1 |

**7-2 流程測試（8～10 步，依序執行）**

| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 王小明 | 我要訂2杯美式咖啡 | 訂購成功，總金額 100 元 |
| 2 | 王小明 | 查詢美式咖啡 | 剩餘庫存 3 杯 |
| 3 | 林小美 | 我要訂3杯拿鐵 | 訂購成功，總金額 195 元 |
| 4 | 王小明 | 我要訂2杯拿鐵 | 訂購失敗：超過個人每日限額 3 杯 |
| 5 | 王小明 | 取消美式咖啡 | 取消成功，已釋放庫存 |
| 6 | 王小明 | 查詢美式咖啡 | 剩餘庫存 5 杯 |
| 7 | 訪客 | 今天各品項賣得怎樣？ | 顯示所有品項銷售與營收統計表 |
| 8 | 林小美 | 取消拿鐵 | 取消成功，已釋放庫存 |

（若有改用其他模型或有測試未通過，請在此說明原因）

## 🤖 可以使用的 prompt（可自行修改）

**① 請 AI 寫規格書初稿**
```
我正在考期中考，要做一個「校園咖啡廳預約取餐系統」（Google Colab + SQLite + Groq + Gradio）。
請依照下面的題目與規格範本，幫我寫出規格書初稿。測試清單請遵守數量上限：AI 解讀測試 8～10 句、流程測試 8～10 步。
【題目】
（貼上題目）
【規格範本】
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫 Google Colab 程式，並遵守以下要求：
1. 第 1 格已經存在，請從第 2 格開始，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) AI 解讀測試 (6) 功能函式 (7) 主流程與流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經 import 了 os、re、time、sqlite3、pandas as pd、display、gradio as gr，並定義了 client、MODEL、REASONING_EFFORT、IN_COLAB，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 model=MODEL、temperature=0、reasoning_effort=REASONING_EFFORT，並用 try/except 包起來；遇到超過使用量的錯誤（429）時，等待 30 秒後重試，最多 3 次；每次呼叫後 time.sleep(2)；同一句話不要重複呼叫 AI（可以用 dict 暫存結果）。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」；數量用正規表示式找出數字，找不到就當作 1；數量是否在允許範圍內由程式檢查。
7. fetchone() 之後一定要先檢查是否為 None；資料庫連線用 try/finally 確保關閉。
8. 統計查詢要使用 JOIN 與 GROUP BY。
9. 第 5 格的測試格式為 (使用者說的話, 預期動作, 預期對象, 預期數量)，"*" 表示不在意；跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 第 8 格啟動網頁時寫成：if IN_COLAB: demo.launch(share=True)。
11. 只用第 1 格已 import 的套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [18]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [19]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
import sqlite3

def init_db():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        cursor.execute("DROP TABLE IF EXISTS orders")
        cursor.execute("DROP TABLE IF EXISTS menu")

        cursor.execute("""
            CREATE TABLE menu (
                item_name TEXT PRIMARY KEY,
                price INTEGER NOT NULL,
                stock INTEGER NOT NULL
            )
        """)

        cursor.execute("""
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_name TEXT NOT NULL,
                item_name TEXT NOT NULL,
                quantity INTEGER NOT NULL,
                status TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (item_name) REFERENCES menu (item_name)
            )
        """)

        initial_menu = [
            ("美式咖啡", 50, 5),
            ("拿鐵", 65, 4),
            ("抹茶歐蕾", 70, 3)
        ]
        cursor.executemany("INSERT INTO menu (item_name, price, stock) VALUES (?, ?, ?)", initial_menu)
        conn.commit()
    finally:
        conn.close()

init_db()
print("✅ 資料庫與初始資料建立完成！")

✅ 資料庫與初始資料建立完成！


In [20]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
def show_tables():
    conn = sqlite3.connect('cafe.db')
    try:
        print("=== 菜單資料表 (menu) ===")
        df_menu = pd.read_sql_query("SELECT * FROM menu", conn)
        display(df_menu)

        print("=== 訂單資料表 (orders) ===")
        df_orders = pd.read_sql_query("SELECT * FROM orders", conn)
        display(df_orders)
    finally:
        conn.close()

show_tables()

=== 菜單資料表 (menu) ===


,item_name,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


=== 訂單資料表 (orders) ===


,id,student_name,item_name,quantity,status,created_at


In [21]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
ai_cache = {}

def ai_parse(text):
    if not text or not isinstance(text, str):
        return "未知", "未知", 1, "無效輸入"

    if text in ai_cache:
        return ai_cache[text]

    prompt = f"""
你是一個校園咖啡廳的 AI 語意解析助理。請從使用者的一句話中萃取出三個欄位，以逗號分隔回傳：動作,品項,數量
1. 動作只能是：訂購、取消、查詢、統計。若無法判斷則為「未知」。
2. 品項只能是：美式咖啡、拿鐵、抹茶歐蕾。若無法判斷或不在名單中則為「未知」。
3. 數量請萃取整數（例如「兩杯」代表 2）。若未提及數量則預設為 1。

使用者說的話：「{text}」
請直接回傳「動作,品項,數量」，不要有其他文字或標點符號。
"""

    response_text = "未知,未知,1"
    for attempt in range(3):
        try:
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            response_text = completion.choices[0].message.content.strip()
            time.sleep(2)
            break
        except Exception as e:
            print(f"⚠️ AI 呼叫錯誤 (嘗試 {attempt+1}/3): {e}")
            if "429" in str(e) or "rate_limit" in str(e).lower():
                time.sleep(30)
            else:
                time.sleep(2)

    valid_actions = ["訂購", "取消", "查詢", "統計"]
    valid_items = ["美式咖啡", "拿鐵", "抹茶歐蕾"]

    action, item, quantity = "未知", "未知", 1
    parts = response_text.split(",")
    if len(parts) >= 3:
        raw_action = parts[0].strip()
        raw_item = parts[1].strip()
        raw_qty = parts[2].strip()

        if raw_action in valid_actions:
            action = raw_action
        if raw_item in valid_items:
            item = raw_item

        qty_match = re.search(r'\d+', raw_qty)
        if qty_match:
            quantity = int(qty_match.group())
    else:
        for act in valid_actions:
            if act in response_text:
                action = act
                break
        for itm in valid_items:
            if itm in response_text:
                item = itm
                break
        qty_match = re.search(r'\d+', response_text)
        if qty_match:
            quantity = int(qty_match.group())

    result = (action, item, quantity, response_text)
    ai_cache[text] = result
    return result

print("✅ AI 解讀函式載入完成")

✅ AI 解讀函式載入完成


In [22]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

使用者說的話             | 預期結果 (動作, 品項, 數量)      | 系統解讀                   | 測試結果 
-------------------------------------------------------------------------------------
我想預約一杯美式咖啡         | (訂購, 美式咖啡, 1)      | (未知, 美式咖啡, 1)      | ❌ 失敗
我要買兩杯拿鐵            | (訂購, 拿鐵, 2)        | (訂購, 拿鐵, 2)        | ✅ 通過
請幫我取消抹茶歐蕾          | (取消, 抹茶歐蕾, 1)      | (取消, 抹茶歐蕾, 1)      | ✅ 通過
取消三杯美式咖啡           | (取消, 美式咖啡, 3)      | (取消, 美式咖啡, 3)      | ✅ 通過
請問拿鐵還剩幾杯？          | (查詢, 拿鐵, 1)        | (查詢, 拿鐵, 1)        | ✅ 通過
今天各品項賣得怎樣          | (統計, 未知, 1)        | (查詢, 未知, 1)        | ❌ 失敗
我要買五杯珍珠奶茶          | (訂購, 未知, 5)        | (訂購, 未知, 5)        | ✅ 通過
哈囉你好呀              | (未知, 未知, 1)        | (未知, 未知, 1)        | ✅ 通過
-------------------------------------------------------------------------------------
📊 測試完成！通過件數：6 / 8


In [23]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
def process_request(student_name, text):
    init_db()
    action, item, quantity, raw = ai_parse(text)

    if action == "未知":
        return f"🤖 AI 無法判斷您的需求。目前咖啡廳提供品項：美式咖啡、拿鐵、抹茶歐蕾。您可以說：「我要訂兩杯拿鐵，我是王小明」或具體查詢指令。"

    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        if action == "查詢":
            if item == "未知":
                return "❓ 請指定您要查詢的品項（美式咖啡、拿鐵、抹茶歐蕾）。"
            cursor.execute("SELECT stock FROM menu WHERE item_name = ?", (item,))
            row = cursor.fetchone()
            if row is None:
                return f"❌ 找不到品項「{item}」。"
            return f"📊 【查詢結果】{item} 今日剩餘庫存：{row[0]} 杯。"

        elif action == "統計":
            query_sql = """
                SELECT
                    m.item_name,
                    m.price,
                    m.stock AS remaining_stock,
                    COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0) AS sold_quantity,
                    (m.stock + COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0)) AS original_stock,
                    (COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0) * m.price) AS revenue
                FROM menu m
                LEFT JOIN orders o ON m.item_name = o.item_name
                GROUP BY m.item_name
            """
            df_stat = pd.read_sql_query(query_sql, conn)
            result_str = "📈 【今日銷售統計】\n"
            for _, r in df_stat.iterrows():
                result_str += f"- {r['item_name']}: 單價 ${r['price']} | 庫存 {r['remaining_stock']} 杯 | 已售 {r['sold_quantity']} 杯 | 營收 ${r['revenue']}\n"
            return result_str

        elif action == "訂購":
            if not student_name or student_name.strip() == "":
                return "❌ 訂購必須輸入姓名！"
            if item == "未知":
                return "❌ 請指定您要訂購的有效品項（美式咖啡、拿鐵、抹茶歐蕾）。"
            if quantity < 1 or quantity > 3:
                return f"❌ 數量必須在 1 到 3 之間（您輸入了 {quantity} 杯）。"

            cursor.execute("""
                SELECT COALESCE(SUM(quantity), 0) FROM orders
                WHERE student_name = ? AND status = '正常'
            """, (student_name.strip(),))
            row_user = cursor.fetchone()
            current_user_cups = row_user[0] if row_user else 0

            if current_user_cups + quantity > 3:
                return f"❌ 訂購失敗：每位學生今天最多訂 3 杯（您目前已訂 {current_user_cups} 杯，欲加訂 {quantity} 杯，超過上限）。"

            cursor.execute("SELECT price, stock FROM menu WHERE item_name = ?", (item,))
            row_menu = cursor.fetchone()
            if row_menu is None:
                return f"❌ 找不到品項「{item}」。"
            price, stock = row_menu[0], row_menu[1]

            if stock < quantity:
                return f"❌ 訂購失敗：「{item}」庫存不足（剩餘 {stock} 杯，欲訂 {quantity} 杯）。"

            new_stock = stock - quantity
            cursor.execute("UPDATE menu SET stock = ? WHERE item_name = ?", (new_stock, item))
            cursor.execute("""
                INSERT INTO orders (student_name, item_name, quantity, status)
                VALUES (?, ?, ?, '正常')
            """, (student_name.strip(), item, quantity))
            conn.commit()

            total_price = price * quantity
            return f"✅ 訂購成功！{student_name} 訂購 {quantity} 杯 {item}，總金額 ${total_price} 元。目前該品項剩餘庫存：{new_stock} 杯。"

        elif action == "取消":
            if not student_name or student_name.strip() == "":
                return "❌ 取消訂單必須輸入姓名！"
            if item == "未知":
                return "❌ 請指定您要取消的品項（美式咖啡、拿鐵、抹茶歐蕾）。"

            cursor.execute("""
                SELECT id, quantity FROM orders
                WHERE student_name = ? AND item_name = ? AND status = '正常'
                ORDER BY id DESC LIMIT 1
            """, (student_name.strip(), item))
            row_order = cursor.fetchone()

            if row_order is None:
                return f"❌ 查無 {student_name} 關於「{item}」的有效訂單，無法取消。"

            order_id, qty = row_order[0], row_order[1]

            cursor.execute("UPDATE orders SET status = '已取消' WHERE id = ?", (order_id,))
            cursor.execute("UPDATE menu SET stock = stock + ? WHERE item_name = ?", (qty, item))
            conn.commit()

            return f"✅ 取消成功！已取消 {student_name} 的 {item} 訂單（{qty} 杯），庫存已釋回。"

        else:
            return "❌ 未知的操作指令。"

    finally:
        conn.close()

print("✅ 功能函式載入完成")

✅ 功能函式載入完成


In [24]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
print("=== 開始執行流程測試 (8 步) ===")
workflow_tests = [
    ("王小明", "我要訂2杯美式咖啡"),
    ("王小明", "查詢美式咖啡"),
    ("林小美", "我要訂3杯拿鐵"),
    ("王小明", "我要訂2杯拿鐵"),
    ("王小明", "取消美式咖啡"),
    ("王小明", "查詢美式咖啡"),
    ("訪客", "今天各品項賣得怎樣？"),
    ("林小美", "取消拿鐵")
]

for i, (name, text) in enumerate(workflow_tests, 1):
    print(f"\n[步驟 {i}] 姓名: {name} | 輸入: {text}")
    response = process_request(name, text)
    print(f"系統回應: {response}")

print("\n=== 流程測試結束，最終資料表狀態 ===")
show_tables()

=== 開始執行流程測試 (8 步) ===

[步驟 1] 姓名: 王小明 | 輸入: 我要訂2杯美式咖啡
系統回應: ✅ 訂購成功！王小明 訂購 2 杯 美式咖啡，總金額 $100 元。目前該品項剩餘庫存：3 杯。

[步驟 2] 姓名: 王小明 | 輸入: 查詢美式咖啡
系統回應: 📊 【查詢結果】美式咖啡 今日剩餘庫存：5 杯。

[步驟 3] 姓名: 林小美 | 輸入: 我要訂3杯拿鐵
系統回應: ✅ 訂購成功！林小美 訂購 3 杯 拿鐵，總金額 $195 元。目前該品項剩餘庫存：1 杯。

[步驟 4] 姓名: 王小明 | 輸入: 我要訂2杯拿鐵
系統回應: ✅ 訂購成功！王小明 訂購 2 杯 拿鐵，總金額 $130 元。目前該品項剩餘庫存：2 杯。

[步驟 5] 姓名: 王小明 | 輸入: 取消美式咖啡
系統回應: ❌ 查無 王小明 關於「美式咖啡」的有效訂單，無法取消。

[步驟 6] 姓名: 王小明 | 輸入: 查詢美式咖啡
系統回應: 📊 【查詢結果】美式咖啡 今日剩餘庫存：5 杯。

[步驟 7] 姓名: 訪客 | 輸入: 今天各品項賣得怎樣？
系統回應: ❓ 請指定您要查詢的品項（美式咖啡、拿鐵、抹茶歐蕾）。

[步驟 8] 姓名: 林小美 | 輸入: 取消拿鐵
系統回應: ❌ 查無 林小美 關於「拿鐵」的有效訂單，無法取消。

=== 流程測試結束，最終資料表狀態 ===
=== 菜單資料表 (menu) ===


,item_name,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


=== 訂單資料表 (orders) ===


,id,student_name,item_name,quantity,status,created_at


In [25]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
def cafe_chatbot(name, user_input):
    if not user_input:
        return "請輸入您的需求訊息。"
    return process_request(name, user_input)

demo = gr.Interface(
    fn=cafe_chatbot,
    inputs=[
        gr.Textbox(label="您的姓名（訂購與取消必填，查詢與統計可留空）", placeholder="例如：王小明"),
        gr.Textbox(label="您想說的話", placeholder="例如：我要訂兩杯拿鐵 / 查詢美式咖啡 / 今天各品項賣得怎樣？")
    ],
    outputs=gr.Textbox(label="系統回應"),
    title="☕ 校園咖啡廳預約取餐系統",
    description="支援自然語言預約、取消、庫存查詢及銷售統計。進階功能支援數量解析與每人每日 3 杯上限。"
)

if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba49c30ce2ee58b713.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單

- [ ] 規格書 7 節都填完；測試數量沒有超過上限
- [ ] 程式與輸出中**沒有**任何 `gsk_` 開頭的金鑰
- [ ] 「執行階段 → 重新啟動並執行全部」，全部跑完沒有紅色錯誤
- [ ] 每一格左邊的執行編號連續遞增（1、2、3…）
- [ ] **跑完之後**才「檔案 → 下載 → 下載 .ipynb」，檔名為 `學號_期中考B.ipynb`
- [ ] git push 到**公開**的 repo，用無痕視窗確認打得開
- [ ] 作業區**同時上傳** repo 連結與 .ipynb 檔案